In [1]:
"""
Central configuration for the NILM Seq2Point pipeline.
Edit paths and hyperparameters here.
"""

import os

# --------------------------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------------------------
UKDALE_H5_PATH = "/kaggle/input/datasets/aashvibudia/ukdale/ukdale.h5"          # path to ukdale.h5
BUILDING = 1                           # building number (Building 1)

DATA_DIR = "data"                      # where cleaned parquet / npz files go
MODEL_DIR = "models"                   # trained model checkpoints
RESULTS_DIR = "results"                # metrics, plots

for d in (DATA_DIR, MODEL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

# --------------------------------------------------------------------------------------
# Meters (NILMTK channel numbers for Building 1, UK-DALE)
# --------------------------------------------------------------------------------------
AGGREGATE_METER = 1

APPLIANCES = {
    "kettle":    {"meter": 10, "on_threshold": 2000.0, "max_power": 3700.0},
    "fridge":    {"meter": 12, "on_threshold": 50.0,   "max_power": 400.0},
    "microwave": {"meter": 13, "on_threshold": 200.0,  "max_power": 3000.0},
}

# --------------------------------------------------------------------------------------
# Resampling
# --------------------------------------------------------------------------------------
SAMPLE_PERIOD = "6s"   # native UK-DALE low-freq sample period

# Max aggregate power considered physically plausible (sensor-error clipping)
AGGREGATE_CLIP_MAX = 15000.0

# --------------------------------------------------------------------------------------
# Seq2Point window
# --------------------------------------------------------------------------------------
WINDOW_LENGTH = 599     # odd number -> well-defined midpoint, ~1hr at 6s

# --------------------------------------------------------------------------------------
# Train / val / test split by calendar date (contiguous time ranges, no shuffling leakage)
# UK-DALE Building 1 spans 2012-11-09 to 2015-01-05.
# These ranges are deliberately chosen so each split contains both active and
# inactive periods for all three appliances (verified against UK-DALE metadata).
# --------------------------------------------------------------------------------------
SPLITS = {
    "train": ("2013-01-01", "2014-08-31"),
    "val":   ("2014-09-01", "2014-10-31"),
    "test":  ("2014-11-01", "2014-12-31"),
}

# --------------------------------------------------------------------------------------
# Window sampling for training (to deal with extreme class imbalance)
# --------------------------------------------------------------------------------------
# Fraction of training windows that must contain an "active" appliance sample
# (i.e. appliance power > on_threshold somewhere in the window). The rest are
# drawn uniformly at random (mostly inactive), preserving realistic prior
# while preventing a training set that is 99% pure zeros.
ACTIVE_WINDOW_FRACTION = 0.5

# Number of training windows per appliance (subsampled from the full set of
# possible windows for tractability)
N_TRAIN_WINDOWS = 200_000
N_VAL_WINDOWS = 20_000
# Test set uses ALL windows in the test range (sequential, for realistic eval)

# --------------------------------------------------------------------------------------
# Training hyperparameters
# --------------------------------------------------------------------------------------
BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 4
RANDOM_SEED = 42

In [3]:
"""
preprocess.py

Loads aggregate + appliance meters from ukdale.h5 (Building 1), aligns them
on a common 6-second time index, cleans the result, and writes a single
parquet file for downstream use.

Run:
    python preprocess.py
"""

import pandas as pd
import numpy as np
import logging


logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


def load_meter(h5_path: str, building: int, meter: int, col_name: str) -> pd.Series:
    """
    Load a single meter's power table from the NILMTK HDF5 file and return
    a pandas Series indexed by UTC timestamp.
    """
    key = f"/building{building}/elec/meter{meter}/table"
    log.info(f"Loading {key} -> '{col_name}'")

    with pd.HDFStore(h5_path, mode="r") as store:
        df = store.select(key)

    # df has columns 'index' (int64 nanosecond epoch) and 'values_block_0' (float array)
    if "index" in df.columns:
        timestamps = pd.to_datetime(df["index"], unit="ns", utc=True)
    else:
        # some NILMTK exports store the timestamp as the dataframe index already
        timestamps = pd.to_datetime(df.index, utc=True)

    values = df["values_block_0"]
    # values_block_0 may be a (N,1) array column -- flatten
    if hasattr(values.iloc[0], "__len__"):
        values = values.apply(lambda x: x[0])

    series = pd.Series(values.values.astype(np.float32), index=timestamps, name=col_name)
    series = series[~series.index.duplicated(keep="first")]
    series = series.sort_index()
    log.info(f"  {col_name}: {len(series):,} raw samples, "
             f"{series.index.min()} -> {series.index.max()}")
    return series


def build_aligned_dataframe() -> pd.DataFrame:
    """
    Load aggregate + all appliance meters, align to a common 6-second grid,
    forward/back fill small gaps, clip implausible values, and remove
    negative power readings.
    """
    series_list = []

    agg = load_meter(UKDALE_H5_PATH, BUILDING, AGGREGATE_METER, "aggregate")
    series_list.append(agg)

    for name, cfg in APPLIANCES.items():
        s = load_meter(UKDALE_H5_PATH, BUILDING, cfg["meter"], name)
        series_list.append(s)

    log.info("Outer-joining all series on timestamp index...")
    df = pd.concat(series_list, axis=1, join="outer")
    df = df.sort_index()
    log.info(f"After outer join: {df.shape}")

    # Resample to a uniform 6-second grid. Native UK-DALE sampling is
    # irregular (~6s but not exact), so resample+mean then fill gaps.
    log.info(f"Resampling to {SAMPLE_PERIOD} grid...")
    df = df.resample(SAMPLE_PERIOD).mean()

    # Fill small gaps (sensor dropouts). Limit the fill so we don't paper
    # over long outages with flat fabricated data -- cap at ~5 minutes
    # (50 samples at 6s).
    max_fill = 50
    df = df.ffill(limit=max_fill).bfill(limit=max_fill)

    # Drop any remaining rows with NaNs (long outages on any channel)
    before = len(df)
    df = df.dropna()
    log.info(f"Dropped {before - len(df):,} rows with unfillable NaNs "
             f"({len(df):,} remain)")

    # Remove negative power readings (sensor artifacts)
    for col in df.columns:
        neg = (df[col] < 0).sum()
        if neg:
            log.info(f"  {col}: clipping {neg:,} negative values to 0")
        df[col] = df[col].clip(lower=0)

    # Clip implausible aggregate spikes (sensor errors)
    spikes = (df["aggregate"] > AGGREGATE_CLIP_MAX).sum()
    if spikes:
        log.info(f"  aggregate: clipping {spikes:,} values above "
                 f"{AGGREGATE_CLIP_MAX}W")
    df["aggregate"] = df["aggregate"].clip(upper=AGGREGATE_CLIP_MAX)

    # Also clip each appliance to its physically plausible max
    for name, cfg in APPLIANCES.items():
        clipped = (df[name] > cfg["max_power"]).sum()
        if clipped:
            log.info(f"  {name}: clipping {clipped:,} values above "
                     f"{cfg['max_power']}W")
        df[name] = df[name].clip(upper=cfg["max_power"])

    log.info(f"Final cleaned dataframe: {df.shape}")
    log.info(f"Date range: {df.index.min()} -> {df.index.max()}")

    return df


def report_appliance_stats(df: pd.DataFrame):
    log.info("\n--- Appliance statistics ---")
    for name, cfg in APPLIANCES.items():
        s = df[name]
        on_pct = (s > cfg["on_threshold"]).mean() * 100
        log.info(
            f"{name:12s} mean={s.mean():8.2f} W  max={s.max():8.1f} W  "
            f"ON%={on_pct:6.3f}  (threshold={cfg['on_threshold']}W)"
        )


def main():
    df = build_aligned_dataframe()
    report_appliance_stats(df)

    out_path = f"{DATA_DIR}/cleaned.parquet"
    df.to_parquet(out_path)
    log.info(f"Saved cleaned dataframe to {out_path}")


if __name__ == "__main__":
    main()

2026-06-14 02:29:08,946 [INFO] Loading /building1/elec/meter1/table -> 'aggregate'
2026-06-14 02:29:14,482 [INFO]   aggregate: 21,837,636 raw samples, 2012-11-09 22:28:15+00:00 -> 2017-04-26 17:32:40+00:00
2026-06-14 02:29:14,484 [INFO] Loading /building1/elec/meter10/table -> 'kettle'
2026-06-14 02:29:19,200 [INFO]   kettle: 18,881,051 raw samples, 2012-11-09 22:28:18+00:00 -> 2017-04-26 17:32:46+00:00
2026-06-14 02:29:19,202 [INFO] Loading /building1/elec/meter12/table -> 'fridge'
2026-06-14 02:29:23,935 [INFO]   fridge: 19,381,298 raw samples, 2012-12-14 22:21:32+00:00 -> 2017-04-26 17:32:51+00:00
2026-06-14 02:29:23,938 [INFO] Loading /building1/elec/meter13/table -> 'microwave'
2026-06-14 02:29:28,470 [INFO]   microwave: 19,406,625 raw samples, 2012-12-14 22:21:33+00:00 -> 2017-04-26 17:33:03+00:00
2026-06-14 02:29:28,472 [INFO] Outer-joining all series on timestamp index...
2026-06-14 02:30:07,336 [INFO] After outer join: (59928837, 4)
2026-06-14 02:30:07,337 [INFO] Resampling to

In [6]:
"""
dataset.py

Builds Seq2Point training/validation/test windows from the cleaned parquet
produced by preprocess.py.

Key design decisions (see chat discussion for rationale):
  - Splits are by contiguous calendar date ranges (config.SPLITS), not random
    row slicing -- avoids both data leakage and "constant target" chunks.
  - Train/val windows are sampled with a target fraction of "active" windows
    (config.ACTIVE_WINDOW_FRACTION) to counter extreme class imbalance,
    while test windows are taken sequentially (every window in the test
    range) for a realistic end-to-end evaluation.
  - Normalization stats (mean/std) are computed on the TRAIN split only and
    reused for val/test.

Run:
    python dataset.py --appliance kettle
"""

import argparse
import json
import logging

import numpy as np
import pandas as pd



logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


def load_cleaned() -> pd.DataFrame:
    return pd.read_parquet(f"{DATA_DIR}/cleaned.parquet")


def slice_split(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    start, end = SPLITS[split_name]
    sliced = df.loc[start:end]
    log.info(f"Split '{split_name}': {start} -> {end}, {len(sliced):,} rows")
    return sliced


def compute_norm_stats(train_df: pd.DataFrame, appliance: str) -> dict:
    """Mean/std computed on the TRAIN split only."""
    return {
        "agg_mean": float(train_df["aggregate"].mean()),
        "agg_std": float(train_df["aggregate"].std()),
        "appliance_mean": float(train_df[appliance].mean()),
        "appliance_std": float(train_df[appliance].std()) or 1.0,
    }


def build_sequential_windows(arr_agg: np.ndarray, arr_app: np.ndarray, window: int):
    """
    Yield (start_index) for every valid window in the array, sequentially.
    Returns the array of valid start indices (covers the whole series).
    """
    n = len(arr_agg)
    n_windows = n - window + 1
    if n_windows <= 0:
        return np.array([], dtype=np.int64)
    return np.arange(n_windows, dtype=np.int64)


def sample_windows(
    arr_app: np.ndarray,
    n_windows: int,
    window: int,
    on_threshold: float,
    active_fraction: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    Sample `n_windows` window-start indices from arr_app such that roughly
    `active_fraction` of them contain at least one sample where
    arr_app > on_threshold (i.e. appliance is ON somewhere in the window).
    """
    n = len(arr_app)
    max_start = n - window
    if max_start <= 0:
        raise ValueError("Series shorter than window length")

    # Precompute, for each position, whether the appliance is active
    active_mask = arr_app > on_threshold

    # Find indices where appliance is active
    active_positions = np.where(active_mask)[0]

    n_active_target = int(n_windows * active_fraction)
    n_inactive_target = n_windows - n_active_target

    starts = []

    # --- Active windows: pick a random active position, then derive a
    # window start such that this position falls inside the window.
    if len(active_positions) > 0:
        chosen_active_pos = rng.choice(active_positions, size=n_active_target, replace=True)
        # window start s must satisfy s <= pos <= s + window - 1
        # => s in [pos - window + 1, pos], clipped to [0, max_start]
        offsets = rng.integers(0, window, size=n_active_target)
        starts_active = np.clip(chosen_active_pos - offsets, 0, max_start)
        starts.append(starts_active)
    else:
        log.warning("No active samples found above threshold in this split! "
                     "All windows will be inactive.")
        n_inactive_target = n_windows

    # --- Inactive (uniform random) windows
    starts_inactive = rng.integers(0, max_start + 1, size=n_inactive_target)
    starts.append(starts_inactive)

    all_starts = np.concatenate(starts)
    rng.shuffle(all_starts)
    return all_starts.astype(np.int64)


def make_windows(df: pd.DataFrame, appliance: str, starts: np.ndarray, window: int,
                  norm_stats: dict):
    """
    Materialize X (windowed aggregate power, normalized) and y (appliance
    power at window midpoint, normalized) for the given start indices.
    """
    agg = df["aggregate"].values.astype(np.float32)
    app = df[appliance].values.astype(np.float32)

    mid_offset = window // 2

    X = np.empty((len(starts), window), dtype=np.float32)
    y = np.empty((len(starts),), dtype=np.float32)

    for i, s in enumerate(starts):
        X[i] = agg[s:s + window]
        y[i] = app[s + mid_offset]

    # Normalize
    X = (X - norm_stats["agg_mean"]) / norm_stats["agg_std"]
    y = (y - norm_stats["appliance_mean"]) / norm_stats["appliance_std"]

    return X, y


def build_dataset_for_appliance(appliance: str):
    cfg = APPLIANCES[appliance]
    df = load_cleaned()

    train_df = slice_split(df, "train")
    val_df = slice_split(df, "val")
    test_df = slice_split(df, "test")

    if len(train_df) < WINDOW_LENGTH or len(val_df) < WINDOW_LENGTH or len(test_df) < WINDOW_LENGTH:
        raise ValueError("One of the splits is shorter than WINDOW_LENGTH. "
                          "Check config.SPLITS against your dataset's date range.")

    norm_stats = compute_norm_stats(train_df, appliance)
    log.info(f"[{appliance}] normalization stats: {norm_stats}")

    rng = np.random.default_rng(RANDOM_SEED)

    # --- Train windows (sampled, balanced for activity) ---
    train_starts = sample_windows(
        train_df[appliance].values, N_TRAIN_WINDOWS, WINDOW_LENGTH,
        cfg["on_threshold"], ACTIVE_WINDOW_FRACTION, rng,
    )
    X_train, y_train = make_windows(train_df, appliance, train_starts, WINDOW_LENGTH, norm_stats)

    # --- Validation windows (sampled, same scheme) ---
    val_starts = sample_windows(
        val_df[appliance].values, N_VAL_WINDOWS, WINDOW_LENGTH,
        cfg["on_threshold"], ACTIVE_WINDOW_FRACTION, rng,
    )
    X_val, y_val = make_windows(val_df, appliance, val_starts, WINDOW_LENGTH, norm_stats)

    # --- Test windows (ALL sequential windows -- realistic evaluation) ---
    test_starts = build_sequential_windows(
        test_df["aggregate"].values, test_df[appliance].values, WINDOW_LENGTH
    )
    log.info(f"[{appliance}] test set: {len(test_starts):,} sequential windows")
    X_test, y_test = make_windows(test_df, appliance, test_starts, WINDOW_LENGTH, norm_stats)

    # Report active-window fraction actually achieved
    on_thresh_norm = (cfg["on_threshold"] - norm_stats["appliance_mean"]) / norm_stats["appliance_std"]
    train_active_frac = (y_train > on_thresh_norm).mean()
    log.info(f"[{appliance}] train active-window fraction (target midpoint > "
             f"threshold): {train_active_frac:.3f}")

    out_prefix = f"{DATA_DIR}/{appliance}"
    np.savez_compressed(
        f"{out_prefix}_train.npz", X=X_train, y=y_train,
    )
    np.savez_compressed(
        f"{out_prefix}_val.npz", X=X_val, y=y_val,
    )
    np.savez_compressed(
        f"{out_prefix}_test.npz", X=X_test, y=y_test,
    )
    with open(f"{out_prefix}_norm_stats.json", "w") as f:
        json.dump(norm_stats, f, indent=2)

    log.info(f"[{appliance}] saved: train={X_train.shape}, val={X_val.shape}, "
             f"test={X_test.shape} -> {out_prefix}_{{train,val,test}}.npz")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--appliance", choices=list(APPLIANCES.keys()) + ["all"],
                         default="all")
    args, _ = parser.parse_known_args()

    targets = list(APPLIANCES.keys()) if args.appliance == "all" else [args.appliance]
    for app in targets:
        log.info(f"=== Building dataset for {app} ===")
        build_dataset_for_appliance(app)


if __name__ == "__main__":
    main()

2026-06-14 02:35:18,223 [INFO] === Building dataset for kettle ===
2026-06-14 02:35:19,371 [INFO] Split 'train': 2013-01-01 -> 2014-08-31, 8,352,454 rows
2026-06-14 02:35:19,373 [INFO] Split 'val': 2014-09-01 -> 2014-10-31, 874,468 rows
2026-06-14 02:35:19,375 [INFO] Split 'test': 2014-11-01 -> 2014-12-31, 870,502 rows
2026-06-14 02:35:19,447 [INFO] [kettle] normalization stats: {'agg_mean': 357.70965576171875, 'agg_std': 420.70001220703125, 'appliance_mean': 14.17298412322998, 'appliance_std': 174.112548828125}
2026-06-14 02:35:20,156 [INFO] [kettle] test set: 869,904 sequential windows
2026-06-14 02:35:22,611 [INFO] [kettle] train active-window fraction (target midpoint > threshold): 0.025
2026-06-14 02:35:56,537 [INFO] [kettle] saved: train=(200000, 599), val=(20000, 599), test=(869904, 599) -> data/kettle_{train,val,test}.npz
2026-06-14 02:35:56,545 [INFO] === Building dataset for fridge ===
2026-06-14 02:35:57,452 [INFO] Split 'train': 2013-01-01 -> 2014-08-31, 8,352,454 rows
2026

In [7]:
"""
model.py

Seq2Point CNN architecture (Zhang et al. 2018 style), adapted for
WINDOW_LENGTH input -> single scalar output (appliance power at the
window midpoint).
"""

import tensorflow as tf
from tensorflow.keras import layers, models




def build_seq2point_model(window_length: int = WINDOW_LENGTH) -> tf.keras.Model:
    inputs = layers.Input(shape=(window_length, 1), name="aggregate_window")

    x = layers.Conv1D(30, 10, activation="relu", padding="same")(inputs)
    x = layers.Conv1D(30, 8, activation="relu", padding="same")(x)
    x = layers.Conv1D(40, 6, activation="relu", padding="same")(x)
    x = layers.Conv1D(50, 5, activation="relu", padding="same")(x)
    x = layers.Conv1D(50, 5, activation="relu", padding="same")(x)

    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="linear", name="appliance_power")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="seq2point")
    return model


if __name__ == "__main__":
    m = build_seq2point_model()
    m.summary()

2026-06-14 02:37:14.435785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781404634.601331      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781404634.648299      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781404635.066389      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781404635.066415      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781404635.066417      58 computation_placer.cc:177] computation placer alr

Model: "seq2point"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aggregate_window (InputLayer)   │ (None, 599, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 599, 30)        │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 599, 30)        │         7,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 599, 40)        │         7,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 599, 50)        │        10,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 599, 50)        │        12,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 29950)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │    30,669,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ appliance_power (Dense)         │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,708,249 (117.14 MB)

 Trainable params: 30,708,249 (117.14 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
"""
train.py

Trains a Seq2Point CNN for a single appliance using the npz files produced
by dataset.py.

Run:
    python train.py --appliance kettle
    python train.py --appliance all
"""


import argparse
import logging

import numpy as np
import tensorflow as tf
log.info(f"GPUs available: {tf.config.list_physical_devices('GPU')}")


logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def load_npz(appliance: str, split: str):
    data = np.load(f"{DATA_DIR}/{appliance}_{split}.npz")
    X, y = data["X"], data["y"]
    X = X.reshape((-1, WINDOW_LENGTH, 1))
    return X, y


def train_appliance(appliance: str):
    log.info(f"=== Training Seq2Point model for '{appliance}' ===")

    X_train, y_train = load_npz(appliance, "train")
    X_val, y_val = load_npz(appliance, "val")
    log.info(f"Train: X={X_train.shape}, y={y_train.shape}")
    log.info(f"Val:   X={X_val.shape}, y={y_val.shape}")

    model = build_seq2point_model()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="mse",
        metrics=["mae"],
    )
    model.summary(print_fn=log.info)

    ckpt_path = f"{MODEL_DIR}/{appliance}_seq2point.keras"
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_loss", save_best_only=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6,
        ),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=2,
    )

    model.save(ckpt_path)
    log.info(f"Saved best model to {ckpt_path}")

    # Save training history for later plotting
    np.savez(f"{MODEL_DIR}/{appliance}_history.npz", **history.history)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--appliance", choices=list(APPLIANCES.keys()) + ["all"],
                         default="all")
    args, _ = parser.parse_known_args()

    targets = list(APPLIANCES.keys()) if args.appliance == "all" else [args.appliance]
    for app in targets:
        train_appliance(app)


if __name__ == "__main__":
    main()

2026-06-14 02:37:30,080 [INFO] GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2026-06-14 02:37:30,085 [INFO] === Training Seq2Point model for 'kettle' ===
2026-06-14 02:37:32,630 [INFO] Train: X=(200000, 599, 1), y=(200000,)
2026-06-14 02:37:32,631 [INFO] Val:   X=(20000, 599, 1), y=(20000,)


2026-06-14 02:37:32,708 [INFO] Model: "seq2point"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aggregate_window (InputLayer)   │ (None, 599, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 599, 30)        │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 599, 30)        │         7,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 599, 40)        │         7,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 599, 50)        │        10,050 │
├─────────────────────────

Epoch 1/20


I0000 00:00:1781404655.923019     124 service.cc:152] XLA service 0x78e33400f610 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781404655.923066     124 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1781404655.923073     124 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1781404656.364424     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1781404661.691685     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


782/782 - 63s - 80ms/step - loss: 1.1330 - mae: 0.2669 - val_loss: 1.0259 - val_mae: 0.2351 - learning_rate: 0.0010
Epoch 2/20
782/782 - 54s - 69ms/step - loss: 0.6589 - mae: 0.1732 - val_loss: 0.9558 - val_mae: 0.2360 - learning_rate: 0.0010
Epoch 3/20
782/782 - 54s - 69ms/step - loss: 0.5124 - mae: 0.1551 - val_loss: 0.9094 - val_mae: 0.2331 - learning_rate: 0.0010
Epoch 4/20
782/782 - 54s - 69ms/step - loss: 0.4070 - mae: 0.1479 - val_loss: 0.7974 - val_mae: 0.2156 - learning_rate: 0.0010
Epoch 5/20
782/782 - 52s - 67ms/step - loss: 0.3519 - mae: 0.1434 - val_loss: 0.8551 - val_mae: 0.2311 - learning_rate: 0.0010
Epoch 6/20
782/782 - 52s - 67ms/step - loss: 0.3078 - mae: 0.1385 - val_loss: 0.9636 - val_mae: 0.2206 - learning_rate: 0.0010
Epoch 7/20
782/782 - 52s - 67ms/step - loss: 0.2231 - mae: 0.1140 - val_loss: 0.8263 - val_mae: 0.1816 - learning_rate: 5.0000e-04
Epoch 8/20
782/782 - 54s - 69ms/step - loss: 0.1608 - mae: 0.0994 - val_loss: 0.7927 - val_mae: 0.1825 - learning_rate

...

Epoch 1/20
782/782 - 62s - 80ms/step - loss: 0.5011 - mae: 0.5131 - val_loss: 0.4050 - val_mae: 0.4451 - learning_rate: 0.0010
Epoch 2/20
782/782 - 55s - 70ms/step - loss: 0.3447 - mae: 0.3994 - val_loss: 0.3368 - val_mae: 0.3959 - learning_rate: 0.0010
Epoch 3/20
782/782 - 55s - 70ms/step - loss: 0.2642 - mae: 0.3396 - val_loss: 0.2895 - val_mae: 0.3484 - learning_rate: 0.0010
Epoch 4/20
782/782 - 55s - 70ms/step - loss: 0.2077 - mae: 0.2980 - val_loss: 0.2883 - val_mae: 0.3442 - learning_rate: 0.0010
Epoch 5/20
782/782 - 53s - 68ms/step - loss: 0.1715 - mae: 0.2722 - val_loss: 0.3086 - val_mae: 0.3605 - learning_rate: 0.0010
Epoch 6/20
782/782 - 53s - 67ms/step - loss: 0.1476 - mae: 0.2530 - val_loss: 0.2966 - val_mae: 0.3489 - learning_rate: 0.0010
Epoch 7/20
782/782 - 55s - 70ms/step - loss: 0.1219 - mae: 0.2287 - val_loss: 0.2809 - val_mae: 0.3261 - learning_rate: 5.0000e-04
Epoch 8/20
782/782 - 55s - 70ms/step - loss: 0.1034 - mae: 0.2118 - val_loss: 0.2796 - val_mae: 0.3249 - le

2026-06-14 03:07:18,995 [INFO] Saved best model to models/fridge_seq2point.keras
2026-06-14 03:07:18,999 [INFO] === Training Seq2Point model for 'microwave' ===
2026-06-14 03:07:21,601 [INFO] Train: X=(200000, 599, 1), y=(200000,)
2026-06-14 03:07:21,602 [INFO] Val:   X=(20000, 599, 1), y=(20000,)


2026-06-14 03:07:21,661 [INFO] Model: "seq2point"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ aggregate_window (InputLayer)   │ (None, 599, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_15 (Conv1D)              │ (None, 599, 30)        │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_16 (Conv1D)              │ (None, 599, 30)        │         7,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 599, 40)        │         7,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 599, 50)        │        10,050 │
├─────────────────────────

Epoch 1/20
782/782 - 63s - 80ms/step - loss: 5.0375 - mae: 0.8492 - val_loss: 4.4931 - val_mae: 0.8175 - learning_rate: 0.0010
Epoch 2/20
782/782 - 55s - 70ms/step - loss: 2.7925 - mae: 0.5326 - val_loss: 3.6720 - val_mae: 0.5702 - learning_rate: 0.0010
Epoch 3/20
782/782 - 53s - 67ms/step - loss: 1.8603 - mae: 0.3954 - val_loss: 3.7439 - val_mae: 0.5504 - learning_rate: 0.0010
Epoch 4/20
782/782 - 52s - 67ms/step - loss: 1.2873 - mae: 0.3319 - val_loss: 3.8664 - val_mae: 0.5519 - learning_rate: 0.0010
Epoch 5/20
782/782 - 53s - 67ms/step - loss: 0.8141 - mae: 0.2533 - val_loss: 3.9612 - val_mae: 0.5414 - learning_rate: 5.0000e-04
Epoch 6/20
782/782 - 52s - 67ms/step - loss: 0.5789 - mae: 0.2135 - val_loss: 3.9452 - val_mae: 0.5332 - learning_rate: 5.0000e-04


2026-06-14 03:12:52,014 [INFO] Saved best model to models/microwave_seq2point.keras


In [9]:
"""
evaluate.py

Evaluates a trained Seq2Point model on the test set:
  - MAE (denormalized, in Watts)
  - SAE (Signal Aggregate Error -- relative total-energy error)
  - Precision / Recall / F1 on ON/OFF state detection (thresholded)

Run:
    python evaluate.py --appliance kettle
    python evaluate.py --appliance all
"""

import argparse
import json
import logging

import numpy as np
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score


logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


def load_npz(appliance: str, split: str):
    data = np.load(f"{DATA_DIR}/{appliance}_{split}.npz")
    X, y = data["X"], data["y"]
    X = X.reshape((-1, WINDOW_LENGTH, 1))
    return X, y


def denormalize(arr: np.ndarray, mean: float, std: float) -> np.ndarray:
    return arr * std + mean


def signal_aggregate_error(y_true_w: np.ndarray, y_pred_w: np.ndarray) -> float:
    """
    SAE = |sum(pred) - sum(true)| / sum(true)
    Measures total energy estimation error (not per-sample accuracy).
    """
    total_true = y_true_w.sum()
    total_pred = y_pred_w.sum()
    if total_true == 0:
        return float("nan")
    return abs(total_pred - total_true) / total_true


def evaluate_appliance(appliance: str):
    log.info(f"=== Evaluating '{appliance}' ===")

    cfg = APPLIANCES[appliance]
    with open(f"{DATA_DIR}/{appliance}_norm_stats.json") as f:
        norm_stats = json.load(f)

    X_test, y_test_norm = load_npz(appliance, "test")
    model = tf.keras.models.load_model(f"{MODEL_DIR}/{appliance}_seq2point.keras")

    y_pred_norm = model.predict(X_test, batch_size=512, verbose=0).flatten()

    # Denormalize to Watts
    y_true_w = denormalize(y_test_norm, norm_stats["appliance_mean"], norm_stats["appliance_std"])
    y_pred_w = denormalize(y_pred_norm, norm_stats["appliance_mean"], norm_stats["appliance_std"])

    # Negative power is unphysical -- clip predictions
    y_pred_w = np.clip(y_pred_w, 0, None)

    mae = np.mean(np.abs(y_true_w - y_pred_w))
    sae = signal_aggregate_error(y_true_w, y_pred_w)

    # ON/OFF detection
    on_thresh = cfg["on_threshold"]
    y_true_state = (y_true_w > on_thresh).astype(int)
    y_pred_state = (y_pred_w > on_thresh).astype(int)

    precision = precision_score(y_true_state, y_pred_state, zero_division=0)
    recall = recall_score(y_true_state, y_pred_state, zero_division=0)
    f1 = f1_score(y_true_state, y_pred_state, zero_division=0)

    results = {
        "appliance": appliance,
        "n_test_samples": int(len(y_true_w)),
        "mae_watts": float(mae),
        "sae": float(sae),
        "on_threshold_watts": on_thresh,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "true_on_fraction": float(y_true_state.mean()),
        "pred_on_fraction": float(y_pred_state.mean()),
    }

    log.info(json.dumps(results, indent=2))

    with open(f"{RESULTS_DIR}/{appliance}_metrics.json", "w") as f:
        json.dump(results, f, indent=2)

    # Save raw predictions for visualization
    np.savez_compressed(
        f"{RESULTS_DIR}/{appliance}_predictions.npz",
        y_true=y_true_w, y_pred=y_pred_w,
    )

    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--appliance", choices=list(APPLIANCES.keys()) + ["all"],
                         default="all")
    args, _ = parser.parse_known_args()

    targets = list(APPLIANCES.keys()) if args.appliance == "all" else [args.appliance]
    all_results = []
    for app in targets:
        all_results.append(evaluate_appliance(app))

    # Summary table
    log.info("\n=== Summary ===")
    log.info(f"{'Appliance':<12} {'MAE (W)':>10} {'SAE':>8} {'Precision':>10} "
             f"{'Recall':>8} {'F1':>8}")
    for r in all_results:
        log.info(f"{r['appliance']:<12} {r['mae_watts']:>10.2f} {r['sae']:>8.3f} "
                 f"{r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1']:>8.3f}")


if __name__ == "__main__":
    main()

2026-06-14 03:12:52,146 [INFO] === Evaluating 'kettle' ===
2026-06-14 03:13:38,906 [INFO] {
  "appliance": "kettle",
  "n_test_samples": 869904,
  "mae_watts": 11.774842262268066,
  "sae": 0.2310476005077362,
  "on_threshold_watts": 2000.0,
  "precision": 0.9760015117157974,
  "recall": 0.6750751535746962,
  "f1": 0.7981148110948003,
  "true_on_fraction": 0.008795223380970774,
  "pred_on_fraction": 0.00608342989571263
}
2026-06-14 03:13:39,150 [INFO] === Evaluating 'fridge' ===
2026-06-14 03:14:25,095 [INFO] {
  "appliance": "fridge",
  "n_test_samples": 869904,
  "mae_watts": 15.335296630859375,
  "sae": 0.0639692023396492,
  "on_threshold_watts": 50.0,
  "precision": 0.8989950047573739,
  "recall": 0.8867325391662312,
  "f1": 0.8928216692169337,
  "true_on_fraction": 0.43116021997829646,
  "pred_on_fraction": 0.4252791112582538
}
2026-06-14 03:14:25,337 [INFO] === Evaluating 'microwave' ===
2026-06-14 03:15:11,476 [INFO] {
  "appliance": "microwave",
  "n_test_samples": 869904,
  "ma

In [12]:
"""
visualize.py

Generates the diagnostic plots discussed:
  - Ground truth vs predicted power, time series overlay (a representative
    window from the test set)
  - Aggregate vs sum-of-disaggregated-appliances vs true aggregate
  - Confusion matrix for ON/OFF detection per appliance
  - Histogram of appliance power distribution (with ON threshold marked)
  - Training/validation loss curves
  - Residual scatter (error vs ground-truth power)

Run:
    python visualize.py --appliance all
"""

import argparse
import json
import logging

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay



logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


def plot_loss_curve(appliance: str):
    hist_path = f"{MODEL_DIR}/{appliance}_history.npz"
    try:
        hist = np.load(hist_path)
    except FileNotFoundError:
        log.warning(f"No history file for {appliance}, skipping loss curve")
        return

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(hist["loss"], label="train")
    axes[0].plot(hist["val_loss"], label="val")
    axes[0].set_title(f"{appliance}: Loss (MSE, normalized)")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(hist["mae"], label="train")
    axes[1].plot(hist["val_mae"], label="val")
    axes[1].set_title(f"{appliance}: MAE (normalized)")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    fig.tight_layout()
    out = f"{RESULTS_DIR}/{appliance}_loss_curve.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    log.info(f"Saved {out}")


def plot_timeseries_overlay(appliance: str, n_samples: int = 1500):
    pred_path = f"{RESULTS_DIR}/{appliance}_predictions.npz"
    try:
        data = np.load(pred_path)
    except FileNotFoundError:
        log.warning(f"No predictions file for {appliance}, skipping timeseries plot")
        return

    y_true, y_pred = data["y_true"], data["y_pred"]

    # Find a window containing appliance activity for a meaningful plot
    cfg = APPLIANCES[appliance]
    active_idx = np.where(y_true > cfg["on_threshold"])[0]
    if len(active_idx) > 0:
        center = active_idx[len(active_idx) // 2]
    else:
        center = len(y_true) // 2

    start = max(0, center - n_samples // 2)
    end = min(len(y_true), start + n_samples)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(y_true[start:end], label="Ground truth", linewidth=1.2)
    ax.plot(y_pred[start:end], label="Predicted", linewidth=1.0, alpha=0.8)
    ax.axhline(cfg["on_threshold"], color="gray", linestyle="--", linewidth=0.8,
               label="ON threshold")
    ax.set_title(f"{appliance}: Ground truth vs predicted power")
    ax.set_xlabel("Sample (6s intervals)")
    ax.set_ylabel("Power (W)")
    ax.legend()
    fig.tight_layout()

    out = f"{RESULTS_DIR}/{appliance}_timeseries.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    log.info(f"Saved {out}")


def plot_confusion_matrix(appliance: str):
    pred_path = f"{RESULTS_DIR}/{appliance}_predictions.npz"
    try:
        data = np.load(pred_path)
    except FileNotFoundError:
        log.warning(f"No predictions file for {appliance}, skipping confusion matrix")
        return

    y_true, y_pred = data["y_true"], data["y_pred"]
    cfg = APPLIANCES[appliance]

    y_true_state = (y_true > cfg["on_threshold"]).astype(int)
    y_pred_state = (y_pred > cfg["on_threshold"]).astype(int)

    cm = confusion_matrix(y_true_state, y_pred_state, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(4, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=["OFF", "ON"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{appliance}: ON/OFF confusion matrix")
    fig.tight_layout()

    out = f"{RESULTS_DIR}/{appliance}_confusion_matrix.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    log.info(f"Saved {out}")


def plot_power_histogram(appliance: str):
    import pandas as pd
    df = pd.read_parquet(f"{DATA_DIR}/cleaned.parquet", columns=[appliance])
    cfg = APPLIANCES[appliance]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(df[appliance], bins=100, log=True)
    ax.axvline(cfg["on_threshold"], color="red", linestyle="--",
               label=f"ON threshold = {cfg['on_threshold']}W")
    ax.set_title(f"{appliance}: power distribution (full dataset)")
    ax.set_xlabel("Power (W)")
    ax.set_ylabel("Count (log scale)")
    ax.legend()
    fig.tight_layout()

    out = f"{RESULTS_DIR}/{appliance}_power_histogram.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    log.info(f"Saved {out}")


def plot_residuals(appliance: str):
    pred_path = f"{RESULTS_DIR}/{appliance}_predictions.npz"
    try:
        data = np.load(pred_path)
    except FileNotFoundError:
        log.warning(f"No predictions file for {appliance}, skipping residual plot")
        return

    y_true, y_pred = data["y_true"], data["y_pred"]
    residual = y_pred - y_true

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(y_true, residual, s=2, alpha=0.2)
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{appliance}: residual (pred - true) vs ground truth power")
    ax.set_xlabel("Ground truth power (W)")
    ax.set_ylabel("Residual (W)")
    fig.tight_layout()

    out = f"{RESULTS_DIR}/{appliance}_residuals.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    log.info(f"Saved {out}")


def plot_aggregate_reconstruction(n_samples: int = 1500):
    """
    Overlay: true aggregate, sum of true appliance powers (+ unmetered
    'other'), and sum of predicted appliance powers, for the test period.
    """
    import pandas as pd
    

    df = pd.read_parquet(f"{DATA_DIR}/cleaned.parquet")
    test_df = df.loc[SPLITS["test"][0]:SPLITS["test"][1]]

    appliance_names = list(APPLIANCES.keys())

    # Load predictions for each appliance (test set, sequential windows)
    preds = {}
    for app in appliance_names:
        try:
            data = np.load(f"{RESULTS_DIR}/{app}_predictions.npz")
            preds[app] = data["y_pred"]
        except FileNotFoundError:
            log.warning(f"No predictions for {app}, skipping aggregate reconstruction")
            return

    n = min(len(v) for v in preds.values())
    n = min(n, len(test_df), n_samples)

    true_agg = test_df["aggregate"].values[:n]
    true_sum_appliances = sum(test_df[app].values[:n] for app in appliance_names)
    pred_sum_appliances = sum(preds[app][:n] for app in appliance_names)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(true_agg, label="True aggregate", linewidth=1.5, color="black")
    ax.plot(true_sum_appliances, label="Sum of true appliance powers", linewidth=1.0)
    ax.plot(pred_sum_appliances, label="Sum of predicted appliance powers",
            linewidth=1.0, alpha=0.8)
    ax.set_title("Aggregate vs reconstructed disaggregation (test period)")
    ax.set_xlabel("Sample (6s intervals)")
    ax.set_ylabel("Power (W)")
    ax.legend()
    fig.tight_layout()

    out = f"{RESULTS_DIR}/aggregate_reconstruction.png"
    fig.savefig(out, dpi=120)
    plt.close(fig)
    log.info(f"Saved {out}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--appliance", choices=list(APPLIANCES.keys()) + ["all"],
                         default="all")
    args, _ = parser.parse_known_args()

    targets = list(APPLIANCES.keys()) if args.appliance == "all" else [args.appliance]

    for app in targets:
        plot_loss_curve(app)
        plot_timeseries_overlay(app)
        plot_confusion_matrix(app)
        plot_power_histogram(app)
        plot_residuals(app)

    if args.appliance == "all":
        plot_aggregate_reconstruction()


if __name__ == "__main__":
    main()

2026-06-14 03:33:24,904 [INFO] Saved results/kettle_loss_curve.png
2026-06-14 03:33:25,115 [INFO] Saved results/kettle_timeseries.png
2026-06-14 03:33:25,309 [INFO] Saved results/kettle_confusion_matrix.png
2026-06-14 03:33:26,632 [INFO] Saved results/kettle_power_histogram.png
2026-06-14 03:33:27,446 [INFO] Saved results/kettle_residuals.png
2026-06-14 03:33:27,694 [INFO] Saved results/fridge_loss_curve.png
2026-06-14 03:33:27,908 [INFO] Saved results/fridge_timeseries.png
2026-06-14 03:33:28,109 [INFO] Saved results/fridge_confusion_matrix.png
2026-06-14 03:33:29,455 [INFO] Saved results/fridge_power_histogram.png
2026-06-14 03:33:30,235 [INFO] Saved results/fridge_residuals.png
2026-06-14 03:33:30,454 [INFO] Saved results/microwave_loss_curve.png
2026-06-14 03:33:30,638 [INFO] Saved results/microwave_timeseries.png
2026-06-14 03:33:30,815 [INFO] Saved results/microwave_confusion_matrix.png
2026-06-14 03:33:32,132 [INFO] Saved results/microwave_power_histogram.png
2026-06-14 03:33:32